# _lib/bronze_cdf — Bronze Change Data Feed reader

Single-function wrapper around `spark.readStream.option("readChangeFeed", "true")`
with consistent options.  Reusable for all Silver tables that consume Bronze CDF.

**Import via `%run ./_lib/bronze_cdf` from the Silver DLT pipeline notebook.**

### Why CDF + apply_changes?

Bronze writes in two modes: `full` (overwrite) and `incremental` (append).  A plain
streaming read on a table that is overwritten fails with a schema-evolution or
source-not-streamable error.  CDF converts overwrites into `delete_row` + `insert_row`
events, and DLT's `apply_changes` declaratively MERGEs those into Silver — no Silver
pipeline state is broken by a Bronze mode switch.  See ADR 0002.

In [ ]:
from pyspark.sql import DataFrame


def read_bronze_cdf(table: str) -> DataFrame:
    """Return a streaming DataFrame reading from a Bronze table's Change Data Feed.

    Args:
        table: Fully-qualified table name, e.g. 'DEMO.STAGING_AZURESTORAGE.order_header'

    Returns:
        A streaming DataFrame with CDF columns (_change_type, _commit_version,
        _commit_timestamp) plus all source columns.

    The caller (DLT apply_changes) will select the relevant columns from this stream
    and route them by _change_type to the target Streaming Table.
    """
    return (
        spark.readStream
        .option("readChangeFeed", "true")
        .option("startingVersion", 0)
        .table(table)
    )


print("bronze_cdf loaded: read_bronze_cdf")